# LTCM 

*Case: Long-Term Capital Management, L.P. (A) [9-200-007].*

# 1. READING

### 1. 
Describe LTCM’s investment strategy with regard to the following aspects:
* Securities traded
* Trading frequency
* Skewness (Do they seek many small wins or a few big hits?)
* Forecasting (What is behind their selection of trades?)

### 2. 
What are LTCM’s biggest advantages over its competitors?

### 3.
The case discusses four types of funding risk facing LTCM:
* collateral haircuts
* repo maturity
* equity redemption
* loan access

The case discusses specific ways in which LTCM manages each of these risks. Briefly discuss
them.

### 4. 

LTCM is largely in the business of selling liquidity and volatility. Describe how LTCM accounts
for liquidity risk in their quantitative measurements.

### 5.

Is leverage risk currently a concern for LTCM?

### 6. 

Many strategies of LTCM rely on converging spreads. LTCM feels that these are almost win/win
situations because of the fact that if the spread converges, they make money. If it diverges, the
trade becomes even more attractive, as convergence is still expected at a future date.

What is the risk in these convergence trades?

***

# 2. Fund Performance and Attribution

### Data

* `ltcm exhibits data.xlsx`, `Exhibit 2`: Gross and net (total) returns of LTCM
* `spy_data.xlsx`: SPY returns and risk-free rate (scaled tbill index)

In [26]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [14]:
file_path_exhibit = "../data/ltcm_exhibits_data.xlsx"
df_exhibit = pd.read_excel(file_path_exhibit, sheet_name="Exhibit 2")

df_exhibit.columns = df_exhibit.iloc[1]
df_exhibit.rename(columns={df_exhibit.columns[0]: 'date'}, inplace=True)

df_exhibit.drop(df_exhibit.index[0:3], inplace=True)

date_str = (
    df_exhibit['date']
    .astype(str)
    .str.strip()
    .str.split().str[0]
)

df_exhibit['date'] = pd.to_datetime(date_str, format="%Y-%m-%d", errors='coerce')
df_exhibit.set_index("date", inplace=True)


In [17]:
file_path_returns = "../data/spy_data.xlsx"
xls_spy_data = pd.ExcelFile(file_path_returns)
df_returns = pd.read_excel(xls_spy_data, sheet_name="total returns", index_col="date")
df_returns.index = df_returns.index.to_period('M').to_timestamp()
df_returns.dropna(axis = 0, inplace = True)
df_returns.head()


df_excess_returns = pd.read_excel(xls_spy_data, sheet_name="excess returns", index_col='date')
df_excess_returns.index = df_excess_returns.index.to_period('M').to_timestamp()

In [21]:
df = df_exhibit.join(df_excess_returns, how='inner')
df

,Fund Capital ($billions),Gross Monthly Performancea,Net Monthly Performanceb,Index of Net Performance,SPY
date,,,,,
1994-03-01,1.1,-0.011,-0.013,0.99,-0.050288
1994-04-01,1.1,0.014,0.008,1,0.007996
1994-05-01,1.2,0.068,0.053,1.05,0.012464
1994-06-01,1.2,-0.039,-0.029,1.02,-0.032782
1994-07-01,1.4,0.116,0.084,1.1,0.028768
1994-08-01,1.5,0.038,0.03,1.14,0.034321
1994-09-01,1.5,-0.004,-0.003,1.13,-0.035039
1994-10-01,1.5,0.01,0.004,1.14,0.024243
1994-11-01,1.6,0.077,0.061,1.21,-0.044441


### 1. Summary stats.

For both the gross and net series of LTCM excess returns, report the annualized 
* mean
* volatility
* Sharpe ratios

Also report the
* skewness
* kurtosis
* 5th quantile

### 2. Compare to SPY

Comment on how these stats compare to SPY and other assets we have seen. 

How much do they differ between gross and net?

In [25]:
#1 and 2
returns=df[['Gross Monthly Performancea', 'Net Monthly Performanceb', 'SPY']]
factor=12

stats=pd.DataFrame(index=returns.columns)
stats["mean"]   =   returns.mean() * factor
stats['vol']    =   returns.std() * np.sqrt(factor)
stats['sharpe'] =   stats['mean'] / stats['vol']
stats['skew']   =   returns.skew()
stats['kurtosis']   =   returns.kurtosis()
stats['VaR']        =   returns.quantile(0.05)
stats


,mean,vol,sharpe,skew,kurtosis,VaR
Gross Monthly Performancea,0.293887,0.136354,2.155321,-0.296428,1.569354,-0.0264
Net Monthly Performanceb,0.20717,0.111904,1.851315,-0.81787,2.905537,-0.0224
SPY,0.154775,0.114073,1.356806,-0.406867,-0.388002,-0.049667


The LTCM strategy earns a higher average monthly return and Sharpe ratio than SPY, with similar or slightly lower volatility. However, returns are negatively skewed and somewhat fat-tailed, which means bigger typical downside moves.

### 3. LFD

Estimate a linear factor decomposition of **net** LTCM excess returns on `SPY` excess returns.

Report
* annualized alpha
* beta
* r-squared

Does LTCM deliver performance beyond `SPY`?

In [33]:
#3
cols = ['Net Monthly Performanceb', 'SPY']
data = df[cols].apply(pd.to_numeric, errors='coerce').dropna()

y = data['Net Monthly Performanceb']
X = sm.add_constant(data['SPY'])

model = sm.OLS(y, X).fit()

alpha = model.params['const'] * factor
beta = model.params['SPY']
r2 = model.rsquared

print("alpha annualized:", alpha)
model.summary()

alpha annualized: 0.18503853217124938


<class 'statsmodels.iolib.summary.Summary'>
"""
                               OLS Regression Results                               
====================================================================================
Dep. Variable:     Net Monthly Performanceb   R-squared:                       0.021
Model:                                  OLS   Adj. R-squared:                  0.002
Method:                       Least Squares   F-statistic:                     1.107
Date:                      Wed, 26 Nov 2025   Prob (F-statistic):              0.298
Time:                              09:55:14   Log-Likelihood:                 107.80
No. Observations:                        53   AIC:                            -211.6
Df Residuals:                            51   BIC:                            -207.7
Df Model:                                 1                                         
Covariance Type:                  nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0154      0.005      3.235      0.002       0.006       0.025
SPY            0.1430      0.136      1.052      0.298      -0.130       0.416
==============================================================================
Omnibus:                       14.933   Durbin-Watson:                   1.802
Prob(Omnibus):                  0.001   Jarque-Bera (JB):               25.141
Skew:                          -0.843   Prob(JB):                     3.47e-06
Kurtosis:                       5.922   Cond. No.                         30.7
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

$$\newcommand{\betalinear}{\beta_{\text{linear}}}
\newcommand{\betaquad}{\beta_{\text{quad}}}
$$

### 4. Nonlinear Exposure

Let's check for non-linear market exposure. Run the following regression on LTCM's **net** excess returns:

$$
\tilde{r}_t^{\text{ltcm}} = \alpha + \betalinear \tilde{r}_t^m + \betaquad \left(\tilde{r}_t^m\right)^2 + \epsilon_t
$$

Report 
* annualized alpha
* the linear and quadratic betas
* r-squared

In [37]:
r=df['SPY']
r_2=r**2
base = pd.DataFrame({'r': r, 'r_2': r_2})

X_2 = sm.add_constant(base)
model_2 = sm.OLS(y, X_2).fit()

alpha = model_2.params['const'] * factor
r2 = model_2.rsquared

print("alpha annualized:", alpha)
model_2.summary()

alpha annualized: 0.21295860164116592


<class 'statsmodels.iolib.summary.Summary'>
"""
                               OLS Regression Results                               
====================================================================================
Dep. Variable:     Net Monthly Performanceb   R-squared:                       0.029
Model:                                  OLS   Adj. R-squared:                 -0.010
Method:                       Least Squares   F-statistic:                    0.7346
Date:                      Wed, 26 Nov 2025   Prob (F-statistic):              0.485
Time:                              10:06:01   Log-Likelihood:                 107.99
No. Observations:                        53   AIC:                            -210.0
Df Residuals:                            50   BIC:                            -204.1
Df Model:                                 2                                         
Covariance Type:                  nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0177      0.006      2.901      0.006       0.005       0.030
r              0.1712      0.144      1.187      0.241      -0.119       0.461
r_2           -2.1870      3.568     -0.613      0.543      -9.354       4.980
==============================================================================
Omnibus:                       16.084   Durbin-Watson:                   1.860
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               27.823
Skew:                          -0.905   Prob(JB):                     9.08e-07
Kurtosis:                       6.053   Cond. No.                         800.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### 5. 

* Does the quadratic market factor do much to increase the overall LTCM variation explained by the market?
* From the regression evidence, does LTCM's market exposure behave as if it is long market options or short market options?
* Should we describe LTCM as being positively or negatively exposed to market volatility?

-The quadratic market factor doesn’t add much  as r-squared only goes from about 0.02 to 0.03, so it barely increases the variation explained.

-The negative coefficient on the quadratic term makes LTCM look more like it is short market options.

-That negative convexity implies LTCM is negatively exposed to market volatility as it tends to benefit in calm markets and suffer when volatility spikes.

### 6. 

Let's try to pinpoint the nature of LTCM's nonlinear exposure. Does it come more from exposure to up-markets or down-markets? Run the following regression on LTCM's net excess returns:

$$
\tilde{r}_t^{\text{ltcm}}  = \alpha + \beta\tilde{r}_t^m + \beta_u \max\left(\tilde{r}_t^m-k_1,0\right) + \beta_d \max\left(k_2 - \tilde{r}_t^m\right) + \epsilon_t
$$

where $k_1= .03$ and $k_2= -.03$. 

Report 
* annualized alpha
* market beta, the **up** and **down** betas
* r-squared

In [42]:
k1=0.03
k2=-0.03

r=df['SPY']
r_k1=(r-k1).apply(lambda x: max(x,0))
r_k2=(k2-r).apply(lambda x: max(x,0))

base_k=pd.DataFrame({'r': r, 'r_k1': r_k1, 'r_k2': r_k2})

X_3=sm.add_constant(base_k)
model_3=sm.OLS(y, X_3).fit()

alpha = model_3.params['const'] * factor

print("alpha annualized:", alpha)
model_3.summary()

alpha annualized: 0.15942135542523977


<class 'statsmodels.iolib.summary.Summary'>
"""
                               OLS Regression Results                               
====================================================================================
Dep. Variable:     Net Monthly Performanceb   R-squared:                       0.050
Model:                                  OLS   Adj. R-squared:                 -0.009
Method:                       Least Squares   F-statistic:                    0.8516
Date:                      Wed, 26 Nov 2025   Prob (F-statistic):              0.472
Time:                              10:20:30   Log-Likelihood:                 108.57
No. Observations:                        53   AIC:                            -209.1
Df Residuals:                            49   BIC:                            -201.3
Df Model:                                 3                                         
Covariance Type:                  nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0133      0.006      2.186      0.034       0.001       0.026
r              0.4387      0.284      1.543      0.129      -0.133       1.010
r_k1          -0.7288      0.636     -1.146      0.258      -2.007       0.550
r_k2           1.0463      1.146      0.913      0.366      -1.257       3.350
==============================================================================
Omnibus:                       18.800   Durbin-Watson:                   1.894
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               36.825
Skew:                          -1.013   Prob(JB):                     1.01e-08
Kurtosis:                       6.546   Cond. No.                         275.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### 7.

* Is LTCM long or short the call-like factor? And the put-like factor?
* Which factor moves LTCM more, the call-like factor, or the put-like factor?
* In the previous problem, you commented on whether LTCM is positively or negatively exposed to market volatility. Using this current regression, does this volatility exposure come more from being long the market's upside? Short the market's downside? Something else?

-It seems it is short the call factor and long the put factor given the beta results for each one. 

-The put option eems to matter more given that the beta is larger. This implies that it drives more the returns of LTCM

-LTCM short volatility porfile comes from mainly being expose to downside moves. It earns positive alpha  in regular markets, but has negative returns in stressed markets